# Well Played, Mauer: OBP and SLG Regression

Which of **OBP** and **SLG** carry the most weight?

In [1]:
# import the necessary packages
import os
import sys
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [2]:
# set up the file paths
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
raw_data_dir = os.path.join(project_root, 'data', 'raw')
processed_data_dir = os.path.join(project_root, 'data', 'processed')

# test the paths
# print(f'Project Root: {project_root}')
# print(f'Raw Data Directory: {raw_data_dir}')
# print(f'Processed Data Directory: {processed_data_dir}')

In [3]:
# read in the csv for all qualified seasons from 2006 - 2015
# data courtesy of stathead
filename = 'mlb_qualified_batters_2006_2015.csv'
csv_path = os.path.join(raw_data_dir, filename)
batters = pd.read_csv(csv_path)
batters.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1495 entries, 0 to 1494
Data columns (total 40 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rk                 1495 non-null   int64  
 1   Player             1495 non-null   object 
 2   Season             1495 non-null   int64  
 3   Age                1495 non-null   int64  
 4   Team               1495 non-null   object 
 5   Lg                 1495 non-null   object 
 6   G                  1495 non-null   int64  
 7   PA                 1495 non-null   int64  
 8   AB                 1495 non-null   int64  
 9   R                  1495 non-null   int64  
 10  H                  1495 non-null   int64  
 11  1B                 1495 non-null   int64  
 12  2B                 1495 non-null   int64  
 13  3B                 1495 non-null   int64  
 14  HR                 1495 non-null   int64  
 15  RBI                1495 non-null   int64  
 16  SB                 1495 

## **OBP** and **SLG** Predicting **RE24**

My goal is determine which of **OBP** and **SLG** carry more weight in terms of a batter's overall contribution to **run expectancy**. The independent, or predictor, variables will be **OBP** and **SLG**. The dependent, or response, variable will be **RE24**, the metric that tracks total net **run expectancy** for a player's plate appearances. I chose **RE24** for that reason, instead of **runs scored** or **RBI**. **Runs scored**, for the most part, don't occur during a batter's plate apperance, unless they hit a **home run**. **RBI** are situational. You can't collect or amass **RBI** unless your teammates are on in front of you. That's not within a player's control. The outcome of their individual plate appearance, however, is mostly within their control.

In short, I'll use **linear regression** to determine an equation of the form:

$$ \hat{\text{RE}24} = a \times \text{OBP} + b \times \text{SLG} + c $$

Here, $a$, $b$, and $c$ are coefficients. What I’m especially interested in is the relative size of $a$ and $b$. Whichever is larger gives insight into which variable (**OBP** or **SLG**) is more strongly associated with **run value**.

I will be using the `statsmodels` package to construct my model.

In [4]:
# set up the independent and dependent variables
X = batters[['OBP', 'SLG']]
X = sm.add_constant(X) # add a constant term
y = batters['RE24']

In [5]:
# construct the model
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   RE24   R-squared:                       0.807
Model:                            OLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     3122.
Date:                Tue, 29 Jul 2025   Prob (F-statistic):               0.00
Time:                        14:50:25   Log-Likelihood:                -5349.3
No. Observations:                1495   AIC:                         1.070e+04
Df Residuals:                    1492   BIC:                         1.072e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       -149.5169      2.279    -65.599      0.0

## Model Interpretations

- $ R^2 = 0.807 $: About 80.7% of the variance in **RE24**, a measure of total run expectancy added by a player across a season, is explained by **OBP** and **SLG**. The remaining 19.3% of the variation likely reflects other factors not included in the model. This could include things such as baserunning, lineup context, or sequencing.
- The equation is as follows:  
  $$ \hat{\text{RE}24} = 269.40 \times \text{OBP} + 155.75 \times \text{SLG} - 149.52 $$
    - The coefficient of **269.40** for **OBP** means that, holding **SLG** constant, for every **0.001 (one point)** increase in **OBP**, the model predicts an increase in total **run expectancy** of about **0.27 runs**.
    - The coefficient of **155.75** for **SLG** means that, holding **OBP** constant, for every **0.001 (one point)** increase in **SLG**, the model predicts an increase in total **run expectancy** of about **0.16 runs**.
    - The constant of **-149.25** at the end of the equation serves to anchor the model. It represents the expected total **run expectancy** for a player who records a **.000 OBP** and **.000 SLG** over a qualified season. Essentially, it represent a player who never gets on base.

In [6]:
# access the coefficients of the model
coeffs = model.params

In [7]:
# assign individual coefficients by name
intercept = coeffs['const']
obp_coef = coeffs['OBP']
slg_coef = coeffs['SLG']

print(f'Using these coefficients, OBP is approximately {obp_coef / slg_coef:.2f} times as valuable as SLG.')

Using these coefficients, OBP is approximately 1.73 times as valuable as SLG.


In [8]:
# find the predicted contribution to run scoring using the linear model
batters['RE24_OBP_SLG'] = obp_coef * batters['OBP'] + slg_coef * batters['SLG'] + intercept

# find the residual
batters['RE24_OBP_SLG_residual'] = batters['RE24'] - batters['RE24_OBP_SLG']

## **OPS** Predicting **RE24**

I should also make a comparison by using **OPS** to predicted **RE24**. Does weighing **OBP** and **SLG** the same really cause that much of a difference?

In [9]:
# set up the independent variable
X = batters[['OPS']]
X = sm.add_constant(X) # add a constant term

In [10]:
# construct the model
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   RE24   R-squared:                       0.796
Model:                            OLS   Adj. R-squared:                  0.796
Method:                 Least Squares   F-statistic:                     5818.
Date:                Tue, 29 Jul 2025   Prob (F-statistic):               0.00
Time:                        14:50:25   Log-Likelihood:                -5392.0
No. Observations:                1495   AIC:                         1.079e+04
Df Residuals:                    1493   BIC:                         1.080e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       -138.2760      1.992    -69.422      0.0

Interestingly enough, building the model with **OPS** doesn't really seem to be *that* off. We don't lose much in the way of explained variance. It is *less*, but not *that much* less. (At least, in terms of $ R^2 $.)

In [11]:
# assign individual coefficients by name
coeffs = model.params
intercept = coeffs['const']
ops_coef = coeffs['OPS']

# find the predicted contribution to run scoring using the linear model
batters['RE24_OPS'] = ops_coef * batters['OPS'] + intercept

# find the residual
batters['RE24_OPS_residual'] = batters['RE24'] - batters['RE24_OPS']

## **AVG** Predicting **RE24**

Just for fun, what do we get when we use **AVG** to predict **RE24**?

In [12]:
# set up the independent variable
X = batters[['BA']]
X = sm.add_constant(X) # add a constant term

In [13]:
# construct the model
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   RE24   R-squared:                       0.347
Model:                            OLS   Adj. R-squared:                  0.346
Method:                 Least Squares   F-statistic:                     792.4
Date:                Tue, 29 Jul 2025   Prob (F-statistic):          3.28e-140
Time:                        14:50:25   Log-Likelihood:                -6261.3
No. Observations:                1495   AIC:                         1.253e+04
Df Residuals:                    1493   BIC:                         1.254e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       -106.4302      4.250    -25.044      0.0

**AVG**, as I expected, performed much worse. There's very little correlation between hitting for **AVG** and run expectancy.

In [14]:
# assign individual coefficients by name
coeffs = model.params
intercept = coeffs['const']
avg_coef = coeffs['BA']

# find the predicted contribution to run scoring using the linear model
batters['RE24_AVG'] = avg_coef * batters['BA'] + intercept

# find the residual
batters['RE24_AVG_residual'] = batters['RE24'] - batters['RE24_AVG']

In [16]:
# export the long dataframe for dashboard usage
filename = 'mlb_qualified_batters_2006_2015_processed.csv'
csv_path = os.path.join(processed_data_dir, filename)
batters.to_csv(csv_path)